In [0]:
%pip install statsmodels

In [0]:
%pip install shap

In [0]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Sustituye con los nombres exactos de tu entorno
CATALOG = "datathon"
SCHEMA = "gcc"
GOLD_TABLE = f"{CATALOG}.{SCHEMA}.gold_cement_features"

# Leer tabla Gold y convertir a Pandas
df_gold = spark.table(GOLD_TABLE).toPandas()

In [0]:
plt.figure(figsize=(12, 8))

# Seleccionar solo columnas numéricas (excluyendo strings como ProductType)
numeric_cols = df_gold.select_dtypes(include=[np.number]).columns
correlation_matrix = df_gold[numeric_cols].corr()

# Generar Heatmap
sns.heatmap(correlation_matrix, annot=False, cmap='coolwarm', vmin=-1, vmax=1)
plt.title("Matriz de Correlación - Features y Targets")
plt.show()

# Imprimir las correlaciones específicas con la resistencia a 28 días
print("Correlación con Strength28:")
print(correlation_matrix['Strength28'].sort_values(ascending=False))

In [0]:
# Definir variables objetivo para excluirlas del cálculo de features
targets = ['Strength1', 'Strength7', 'Strength28']

# Excluir targets y columnas de metadatos/strings
feature_cols = [
    col for col in numeric_cols 
    if col not in targets and not col.startswith("_")
]

# Filtrar df para VIF (statsmodels requiere que no haya nulos)
X = df_gold[feature_cols].dropna()

# Calcular VIF
vif_data = pd.DataFrame()
vif_data["Feature"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(len(X.columns))]

# Ordenar de mayor a menor multicolinealidad
vif_data = vif_data.sort_values(by="VIF", ascending=False).reset_index(drop=True)
display(vif_data)

In [0]:
import pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor

# 1. Definir las variables a eliminar basados en el diagnóstico VIF previo
columnas_a_eliminar = [
    'pt_other',                        # Evitar la trampa de la variable ficticia (Dummy Variable Trap)
    'Mineral_Phase_C',                 # Romper la multicolinealidad de los datos composicionales
    'Chemical_Composition_B_missing'   # Eliminar redundancia en los indicadores de nulos
]

# 2. Excluir estas columnas de nuestras features (asumiendo que 'X' era tu DataFrame original con las features)
features_limpias = [col for col in X.columns if col not in columnas_a_eliminar]
X_clean = X[features_limpias].dropna()

# 3. Recalcular el VIF
vif_data_clean = pd.DataFrame()
vif_data_clean["Feature"] = X_clean.columns
vif_data_clean["VIF"] = [variance_inflation_factor(X_clean.values, i) for i in range(len(X_clean.columns))]

# 4. Mostrar los nuevos resultados
vif_data_clean = vif_data_clean.sort_values(by="VIF", ascending=False).reset_index(drop=True)
display(vif_data_clean)